In [1]:
import torch
from models.eegnet.model import EEGNet

# Initialize EEGNet with its accepted parameters
model = EEGNet(
    n_classes=2,               # Number of classes
    electrode_channels=64,     # Number of EEG channels
    sample_length=128,         # Number of time samples
    dropout_rate=0.5,          # Dropout probability
    kernel_length=32,          # Temporal convolution kernel length
    f1=8,                      # Number of temporal filters
    d=2,                       # Number of spatial filters per temporal filter
    f2=16                      # Pointwise convolution filters
)

print(model)

EEGNet(
  (block1_conv1): Conv2d(1, 8, kernel_size=(1, 32), stride=(1, 1), padding=same, bias=False)
  (block1_bn1): BatchNorm2d(8, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
  (block1_conv2): Conv2d(8, 16, kernel_size=(64, 1), stride=(1, 1), groups=8, bias=False)
  (block1_bn2): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
  (block1_elu): ELU(alpha=1.0)
  (block1_pool): AvgPool2d(kernel_size=(1, 4), stride=(1, 4), padding=0)
  (block1_dropout): Dropout(p=0.5, inplace=False)
  (block2_conv1): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=same, groups=16, bias=False)
  (block2_conv2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (block2_bn): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
  (block2_elu): ELU(alpha=1.0)
  (block2_pool): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
  (block2_dropout): Dropout(p=0.5, inplace=False)
  (flatten): Flatten(start

# EEGNet Baseline Pipeline
This notebook establishes the formal baseline for Motor Imagery classification using EEGNet. 
It uses our structured workflow to log the dataset details and hyperparameters to TensorBoard.

In [2]:
from functools import partial

import torch
import torch.nn as nn
import torch.optim as optim
from moabb.datasets import BNCI2014_001, Gao2026
import datetime

# Import custom tools
from tools.dataloading import get_motorimagery_loaders
from tools.preprocessing import apply_mne_ica
from tools.training import Trainer
from models.eegnet.model import EEGNet

In [3]:
# 1. Pipeline Configuration
dataset = BNCI2014_001()

# Setting up standard industry parameters for testing a baseline
hyperparams = {
    "dataset": type(dataset).__name__,
    "split_mode": "loso",
    "test_subject_id": 1,
    "batch_size": 32,
    "sample_frequency": 250,
    "learning_rate": 0.001,
    "epochs": 300,
    "patience": 50,
    "dropout_rate": 0.5,
    "kernel_length": 64,
    "f1": 8,
    "d": 2,
    "f2": 16,
    "preprocessing": "ICA"
}

# Add dynamic timestamp string to organize logs
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_dir = f"models/eegnet/results/baseline_{hyperparams['dataset']}_{hyperparams['split_mode']}_{timestamp}"

In [5]:
custom_ica = partial(apply_mne_ica, n_components=15)


# 2. Data Loading
print(
    f"Loading {hyperparams['dataset']} with {hyperparams['split_mode'].upper()} split for Subject {hyperparams['test_subject_id']}...")

train_loader, test_loader, metadata = get_motorimagery_loaders(
    dataset,
    preprocessing_fn=None,
    sample_frequency=hyperparams["sample_frequency"],
    test_subject_id=hyperparams["test_subject_id"],
    batch_size=hyperparams["batch_size"],
    split_mode=hyperparams["split_mode"]
)

# Extract dynamic dimensions from the dataset that the model needs
hyperparams["n_classes"] = metadata["n_classes"]
hyperparams["electrode_channels"] = metadata["electrode_channels"]
hyperparams["sample_length"] = metadata["sample_length"]

print(
    f"Data ready. Found {hyperparams['n_classes']} classes across {hyperparams['electrode_channels']} channels.")

Choosing from all possible events


Loading BNCI2014_001 with LOSO split for Subject 1...


Split Mode: LOSO | Test Subject: 1
Training on 4608 trials from 8 subjects
Testing on 576 trials from 1 subject
Input Shape for Model: torch.Size([32, 1, 22, 1001])
Batch size: 32 | Nº Channels: 22 | Sample length: 1001
Data ready. Found 4 classes across 22 channels.


In [27]:
# 3. Model Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = EEGNet(
    n_classes=hyperparams["n_classes"],
    electrode_channels=hyperparams["electrode_channels"],
    sample_length=hyperparams["sample_length"],
    kernel_length=hyperparams["kernel_length"],
    dropout_rate=hyperparams["dropout_rate"],
    f1=hyperparams["f1"],
    d=hyperparams["d"],
    f2=hyperparams["f2"]
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=hyperparams["learning_rate"])

Using device: cuda


In [7]:
# 4. Training and Evaluation Tracking
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    log_dir=log_dir,
    experiment_config=hyperparams
)

trainer.train(
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=hyperparams["epochs"],
    patience=hyperparams["patience"]
)

Starting training on cuda...
Logging TensorBoard to: models/eegnet/results/baseline_BNCI2014_001_loso_20260530-212910


/home/carlos/.conda/envs/ml/lib/python3.13/site-packages/torch/nn/modules/conv.py:543: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1027.)
  return F.conv2d(


Epoch [1/300] | Train Loss: 1.3763 | Test Loss: 1.3604 | Test Acc: 28.82%
Epoch [10/300] | Train Loss: 1.1437 | Test Loss: 0.9969 | Test Acc: 61.98%
Epoch [20/300] | Train Loss: 1.0556 | Test Loss: 0.9981 | Test Acc: 60.42%
Epoch [30/300] | Train Loss: 1.0190 | Test Loss: 0.9121 | Test Acc: 64.06%
Epoch [40/300] | Train Loss: 1.0018 | Test Loss: 1.0016 | Test Acc: 58.68%
Epoch [50/300] | Train Loss: 0.9613 | Test Loss: 0.9174 | Test Acc: 63.37%
Epoch [60/300] | Train Loss: 0.9569 | Test Loss: 0.9765 | Test Acc: 59.90%
Epoch [70/300] | Train Loss: 0.9427 | Test Loss: 0.9624 | Test Acc: 62.85%
Epoch [80/300] | Train Loss: 0.9410 | Test Loss: 1.0466 | Test Acc: 55.03%
Epoch [90/300] | Train Loss: 0.9217 | Test Loss: 0.9058 | Test Acc: 63.72%
Epoch [100/300] | Train Loss: 0.9200 | Test Loss: 0.9447 | Test Acc: 61.11%
Epoch [110/300] | Train Loss: 0.9049 | Test Loss: 0.9648 | Test Acc: 60.07%
Epoch [120/300] | Train Loss: 0.8980 | Test Loss: 1.0484 | Test Acc: 57.12%
Early stopping triggere

In [15]:
next(iter(train_loader))[0].shape

torch.Size([32, 1, 22, 1001])

In [21]:
model_to_export = EEGNet(
    n_classes=hyperparams["n_classes"],
    electrode_channels=hyperparams["electrode_channels"],
    sample_length=hyperparams["sample_length"],
    kernel_length=hyperparams["kernel_length"],
    dropout_rate=hyperparams["dropout_rate"],
    f1=hyperparams["f1"],
    d=hyperparams["d"],
    f2=hyperparams["f2"]
)

model_to_export.load_state_dict(torch.load(f"{log_dir}/best_model.pth"))

model_to_export.eval()

dummy_input = torch.randn(
    hyperparams["batch_size"], 1, hyperparams["electrode_channels"], hyperparams["sample_length"])


onnx_file_path = f"{log_dir}/eegnet_baseline.onnx"

torch.onnx.export(
    model_to_export,
    (dummy_input,),
    onnx_file_path,
    dynamo=True,
    verbose=True,
    input_names=["input"],
    output_names=["output"],
    # dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)

[torch.onnx] Obtain model graph for `EEGNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EEGNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.8.0+cu129',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[32,1,22,1001]>
            ),
            outputs=(
                %"output"<FLOAT,[32,4]>
            ),
            initializers=(
                %"block1_conv1.weight"<FLOAT,[8,1,1,64]>{Tensor(...)},
                %"block1_conv2.weight"<FLOAT,[16,1,22,1]>{Tensor(...)},
                %"block2_conv1.weight"<FLOAT,[16,1,1,16]>{TorchTensor(...)},
                %"block2_conv2.weight"<FLOAT,[16,16,1,1]>{Tensor(...)},
                %"dense.weight"<FLOAT,[4,496]>{TorchTensor(...)},
                %"dense.bias"<FLOAT,[4]>{TorchTensor<FLOAT,[4]>(Parameter containing: tensor([-0.1299,  0.0667, -0.0015, -0.0450], requires_grad=True), name='dense.bi

In [23]:
import onnx
from onnx import checker


model_proto = onnx.load(onnx_file_path)
checker.check_model(model_proto)
checker.check_graph(model_proto.graph)

In [ ]:
# Quantise model using ppq

model.load_state_dict(torch.load(f"{log_dir}/best_model.pth"))
model.eval()
print(model)

EEGNet(
  (block1_conv1): Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=same, bias=False)
  (block1_bn1): BatchNorm2d(8, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
  (block1_conv2): Conv2d(8, 16, kernel_size=(22, 1), stride=(1, 1), groups=8, bias=False)
  (block1_bn2): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
  (block1_elu): ELU(alpha=1.0)
  (block1_pool): AvgPool2d(kernel_size=(1, 4), stride=(1, 4), padding=0)
  (block1_dropout): Dropout(p=0.5, inplace=False)
  (block2_conv1): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=same, groups=16, bias=False)
  (block2_conv2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (block2_bn): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
  (block2_elu): ELU(alpha=1.0)
  (block2_pool): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
  (block2_dropout): Dropout(p=0.5, inplace=False)
  (flatten): Flatten(start

In [29]:
from esp_ppq import QuantizationSettingFactory
from esp_ppq.api import espdl_quantize_torch


    ___________ ____        ____  ____  ____
   / ____/ ___// __ \      / __ \/ __ \/ __ \
  / __/  \__ \/ /_/ /_____/ /_/ / /_/ / / / /
 / /___ ___/ / ____/_____/ ____/ ____/ /_/ /
/_____//____/_/         /_/   /_/    \___\_\




In [35]:
BATCHSIZE = 4
ESPDL_MODEL_PATH = f"{log_dir}/eegnet_baseline_espdl.espdl"
INPUT_SHAPE = [1, hyperparams["electrode_channels"],
               hyperparams["sample_length"]]

TARGET = "esp32s3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_BITS = 8


def load_calibration_dataset(n_samples: int, dataloader: torch.utils.data.DataLoader) -> torch.utils.data.TensorDataset:
    features = [dataloader.dataset[i][0]
                # Remove labels for calibration dataset
                for i in range(n_samples)]
    features = torch.stack(features)

    return torch.utils.data.TensorDataset(features)


def collate_fn(batch: torch.Tensor) -> torch.Tensor:
    return batch[0].to(DEVICE)


# create a setting for quantizing your network with ESPDL.
quant_setting = QuantizationSettingFactory.espdl_setting()


# Load training data for creating a calibration dataloader.
calibration_dataset = load_calibration_dataset(
    n_samples=BATCHSIZE * 2, dataloader=train_loader)

calibration_dataloader = torch.utils.data.DataLoader(
    dataset=calibration_dataset, batch_size=BATCHSIZE, shuffle=False)


# quantize your model.
quant_ppq_graph = espdl_quantize_torch(
    model=model,
    espdl_export_file=ESPDL_MODEL_PATH,
    calib_dataloader=calibration_dataloader,
    calib_steps=8,
    input_shape=[1] + INPUT_SHAPE,
    target=TARGET,
    num_of_bits=NUM_BITS,
    collate_fn=collate_fn,
    setting=quant_setting,
    device=DEVICE,
    error_report=False,
    skip_export=False,
    export_test_values=True,
    verbose=1,
)

[23:25:29] ConvTranspose Decomposition Pass Running ... Finished.
[23:25:29] PPQ Quantization Fusion Pass Running ...       Finished.
[23:25:30] PPQ Quantize Simplify Pass Running ...         Finished.
[23:25:30] PPQ Parameter Quantization Pass Running ...    Finished.
[23:25:30] PPQ Runtime Calibration Pass Running ...       

Calibration Progress(Phase 2): 100%|██████████| 8/8 [00:00<00:00, 276.74it/s]

Finished.
[23:25:30] PPQ Quantization Alignment Pass Running ...    Finished.
[23:25:30] PPQ Passive Parameter Quantization Running ... Finished.
--------- Network Snapshot ---------
Num of Op:                    [10]
Num of Quantized Op:          [10]
Num of Variable:              [20]
Num of Quantized Var:         [20]
------- Quantization Snapshot ------
Num of Quant Config:          [29]
ACTIVATED:                    [15]
OVERLAPPED:                   [10]
PASSIVE:                      [4]
Network Quantization Finished.


In [38]:
f"{log_dir}/best_model.pth"

'models/eegnet/results/baseline_BNCI2014_001_loso_20260530-212910/best_model.pth'

In [40]:
import torch
# ensure correct import for setting
from esp_ppq import QuantizationSettingFactory
from esp_ppq.api import espdl_quantize_torch, ENABLE_CUDA_KERNEL

BATCHSIZE = 4
ESPDL_MODEL_PATH = f"{log_dir}/eegnet_baseline_espdl.espdl"
INPUT_SHAPE = [1, hyperparams["electrode_channels"],
               hyperparams["sample_length"]]

TARGET = "esp32s3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_BITS = 8


def load_calibration_dataset(n_samples: int, dataloader: torch.utils.data.DataLoader) -> torch.utils.data.TensorDataset:
    features = [dataloader.dataset[i][0] for i in range(n_samples)]
    features = torch.stack(features)
    return torch.utils.data.TensorDataset(features)


def collate_fn(batch: torch.Tensor) -> torch.Tensor:
    return batch[0].to(DEVICE)


# 1. Create default ESP-DL quantization settings
quant_setting = QuantizationSettingFactory.espdl_setting()

# 2. Enable and Configure Quantization-Aware Training (TQT)
quant_setting.tqt_optimization = True
# Total training/fine-tuning steps
quant_setting.tqt_optimization_setting.steps = 500
# Fine-tuning learning rate
quant_setting.tqt_optimization_setting.lr = 1e-5
quant_setting.tqt_optimization_setting.collecting_device = DEVICE
# Pulls exponents closer to powers of 2 for ESP32-S3
quant_setting.tqt_optimization_setting.int_lambda = 0.25

# 3. Prepare dataset (Increase samples to supply your fine-tuning steps properly)
# Note: For QAT, ideally use a broader slice of training data or your entire loader.
calibration_dataset = load_calibration_dataset(
    n_samples=BATCHSIZE * 32, dataloader=train_loader
)

calibration_dataloader = torch.utils.data.DataLoader(
    # Shuffle for better gradient descent
    dataset=calibration_dataset, batch_size=BATCHSIZE, shuffle=True
)

# 4. Quantize and Fine-tune your model using QAT
# Wrapping with ENABLE_CUDA_KERNEL accelerates PPQ's backward pass if using CUDA

quant_ppq_graph = espdl_quantize_torch(
    model=model,
    espdl_export_file=ESPDL_MODEL_PATH,
    calib_dataloader=calibration_dataloader,
    calib_steps=32,               # Needs to match or be close to your TQT steps setup
    # Note: ESP-DL runtime strictly expects batch size 1 on-chip
    input_shape=[1] + INPUT_SHAPE,
    target=TARGET,
    num_of_bits=NUM_BITS,
    collate_fn=collate_fn,
    setting=quant_setting,
    device=DEVICE,
    error_report=True,             # Turned on to observe QAT improvement metrics
    skip_export=False,
    export_test_values=True,
    verbose=1,
)

[23:39:35] ConvTranspose Decomposition Pass Running ... Finished.
[23:39:36] PPQ Quantization Fusion Pass Running ...       Finished.
[23:39:36] PPQ Quantize Simplify Pass Running ...         Finished.
[23:39:36] PPQ Parameter Quantization Pass Running ...    Finished.
[23:39:36] PPQ Runtime Calibration Pass Running ...       

Calibration Progress(Phase 2): 100%|██████████| 32/32 [00:00<00:00, 516.02it/s]


Finished.
[23:39:36] PPQ Quantization Alignment Pass Running ...    Finished.
[23:39:36] PPQ Passive Parameter Quantization Running ... Finished.
[23:39:36] ESP-PPQ TQT Optimization Running ...           
Check following parameters:
Is Scale Trainable:        True
Interested Layers:         []
Num of blocks:             2
Learning Rate:             1e-05
Steps:                     500
Gamma:                     0.0
int_lambda:                0.25

# Block [1 / 2]: [/block1_conv1/Conv -> /block2_conv1/Conv]


# Tuning Procedure : 100%|██████████| 500/500 [00:04<00:00, 114.70it/s]


# Tuning Finished  : (0.5792 -> 0.3796) [Block Loss]

# Block [2 / 2]: [/block2_conv2/Conv -> /dense/Gemm]


...

# Tuning Finished  : (0.3950 -> 0.3769) [Block Loss]

Finished.
[23:39:46] PPQ Passive Parameter Quantization Running ... Finished.
--------- Network Snapshot ---------
Num of Op:                    [10]
Num of Quantized Op:          [10]
Num of Variable:              [20]
Num of Quantized Var:         [20]
------- Quantization Snapshot ------
Num of Quant Config:          [29]
ACTIVATED:                    [15]
OVERLAPPED:                   [10]
PASSIVE:                      [4]
Network Quantization Finished.


Analysing Graphwise Quantization Error(Phrase 1):: 100%|██████████| 8/8 [00:00<00:00, 74.41it/s]
Analysing Graphwise Quantization Error(Phrase 2):: 100%|██████████| 8/8 [00:00<00:00, 95.46it/s]


Layer               | NOISE:SIGNAL POWER RATIO 
/dense/Gemm:        | ████████████████████ | 267.377%
/block1_conv1/Conv: | ██████               | 145.870%
/block2_conv2/Conv: | ██                   | 118.100%
/block2_conv1/Conv: | █                    | 102.668%
/block1_conv2/Conv: |                      | 96.989%


Analysing Layerwise quantization error:: 100%|██████████| 5/5 [00:00<00:00, 65.07it/s]


Layer               | NOISE:SIGNAL POWER RATIO 
/block1_conv1/Conv: | ████████████████████ | 48.608%
/block2_conv1/Conv: | █                    | 2.751%
/block1_conv2/Conv: |                      | 1.956%
/block2_conv2/Conv: |                      | 1.316%
/dense/Gemm:        |                      | 0.847%


In [43]:
import torch
from tqdm import tqdm
# Force the exact esp_ppq executor engine
from esp_ppq.executor import TorchExecutor


def evaluate_model(model_or_graph, dataloader, is_ppq=False, device="cpu"):
    """
    Evaluates both PyTorch FP32 models and esp_ppq INT8 simulation graphs.
    """
    if not is_ppq:
        model_or_graph.eval()
        model_or_graph.to(device)
    else:
        # Instantiate the correct esp_ppq backend executor engine
        executor = TorchExecutor(graph=model_or_graph, device=device)

    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    # Standard evaluation objective
    criterion = torch.nn.CrossEntropyLoss()

    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Evaluating", leave=False):
            inputs = inputs.to(device)
            labels = labels.to(device)

            # --- Inference Step ---
            if is_ppq:
                # esp_ppq's executor can take inputs as a sequence list
                # corresponding to your graph's expected inputs
                outputs = executor(inputs=[inputs])[0]
            else:
                outputs = model_or_graph(inputs)

            # --- Metric Calculations ---
            loss = criterion(outputs, labels)
            total_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs, 1)
            correct_predictions += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    avg_loss = total_loss / total_samples
    accuracy = (correct_predictions / total_samples) * 100

    return avg_loss, accuracy


# ==========================================
# EXECUTION PIPELINE
# ==========================================

# 1. Run evaluation on your original PyTorch baseline model
print("--- Evaluating Baseline Float32 Model ---")
fp32_loss, fp32_acc = evaluate_model(
    model_or_graph=model,
    dataloader=test_loader,
    is_ppq=False,
    device=DEVICE
)
print(f"FP32 Model -> Loss: {fp32_loss:.4f}, Accuracy: {fp32_acc:.2f}%")


# 2. Run evaluation on your fine-tuned esp_ppq quantization graph output
print("\n--- Evaluating esp_ppq Quantized (TQT INT8) Simulated Model ---")
int8_loss, int8_acc = evaluate_model(
    model_or_graph=quant_ppq_graph,  # graph returned from espdl_quantize_torch
    dataloader=test_loader,
    is_ppq=True,
    device=DEVICE
)
print(f"INT8 Model -> Loss: {int8_loss:.4f}, Accuracy: {int8_acc:.2f}%")


# 3. Structural delta analysis
accuracy_drop = fp32_acc - int8_acc
print("\n" + "="*50)
print(f"esp_ppq Deployment Analytics Summary:")
print(f"  Target Platform:     {TARGET.upper()}")
print(f"  Accuracy Deviation:  {accuracy_drop:+.2f}%")
print("="*50)

--- Evaluating Baseline Float32 Model ---


FP32 Model -> Loss: 0.8569, Accuracy: 67.36%

--- Evaluating esp_ppq Quantized (TQT INT8) Simulated Model ---


INT8 Model -> Loss: 1.0071, Accuracy: 62.15%

esp_ppq Deployment Analytics Summary:
  Target Platform:     ESP32S3
  Accuracy Deviation:  +5.21%
